# Unstain ↔ H&E CycleGAN training

This notebook uses two generators (`A→B`, `B→A`) and two PatchGAN discriminators. Training domains are sampled independently, so registration and paired pixel losses are not used. Corresponding validation crops and raw SSIM are diagnostic only and never contribute to the optimization objective.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from cyclegan_core import (
    CycleGANTrainer, build_dataloaders, denormalize, seed_everything
)

## Parameters

`input_size=512` means a native 512×512 crop from the 0.5 MPP source. No 1024→512 downsampling is performed.

In [ ]:
params = {
    'seed': 42,
    'gpu_index': 1,
    'data_dir': Path('../../data/HnE_n_UNStaining/patch_dataset_mpp05_2048'),
    'output_dir': Path('../../results/Unstain2HnE_cyclegan_v1'),
    'checkpoint_dir': Path('../../model/Unstain2HnE_cyclegan_v1'),
    'image_ext': 'png',
    'image_max_count': 30000,
    'original_size': 2048,
    'input_size': 512,       # native 0.5 MPP crop
    'batch_size': 2,         # two generators need more memory than the prior U-Net
    'num_epochs': 200,
    'decay_start_epoch': 100,
    'val_fraction': 0.10,
    'preload_images': True,
    'max_cache_gib': 64,
    'od_background_threshold': 0.98,
    'od_quantile': 0.995,
    'od_calibration_images': 256,
    'min_crop_tissue_fraction': 0.10,
    'crop_retry_count': 10,
    'ngf': 32,
    'ndf': 64,
    'residual_blocks': 6,
    'lr': 2e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'lambda_cycle': 10.0,
    'lambda_identity': 5.0,
    'pool_size': 50,
    'preview_count': 2,
    'save_every': 10,
    'resume_checkpoint': None,
}

seed_everything(params['seed'])
if torch.cuda.is_available():
    device = torch.device(f"cuda:{params['gpu_index']}")
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device('cpu')
print('device:', device)

## Data

Domain A is deterministic grayscale optical density repeated over three channels. Domain B is RGB H&E. During training, A and B filenames, crop coordinates, and augmentations are all independent. White-only crops are avoided in both domains.

In [ ]:
train_loader, val_loader, od_max = build_dataloaders(params)

real_a, real_b = next(iter(train_loader))
fig, axes = plt.subplots(2, min(4, len(real_a)), figsize=(14, 7), squeeze=False)
for i in range(axes.shape[1]):
    axes[0, i].imshow(denormalize(real_a[i]).permute(1, 2, 0).numpy(), cmap='gray')
    axes[0, i].set_title('Domain A: Unstain OD')
    axes[1, i].imshow(denormalize(real_b[i]).permute(1, 2, 0).numpy())
    axes[1, i].set_title('Independent Domain B: H&E')
    axes[0, i].axis('off')
    axes[1, i].axis('off')
plt.tight_layout()

## Models and training

The generator objective is `GAN(A→B) + GAN(B→A) + 10×cycle + 5×identity`. There is no L1, SSIM, gradient, or blurred loss between generated H&E and registered real H&E.

In [ ]:
trainer = CycleGANTrainer(params, train_loader, val_loader, od_max, device)

In [ ]:
trainer.fit()

## Quick validation preview

`raw_paired_ssim` is printed only to observe translation quality. It is intentionally excluded from training and best-cycle checkpoint selection.

In [ ]:
metrics, preview = trainer.validate()
print(metrics)
trainer.save_preview(max(trainer.start_epoch - 1, 0), preview)